In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
import pytz
from tqdm import tqdm

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-11-18 16:52:17.605055


### Functions

In [3]:
def yield_to_model(list_response_model_name):
    # logic
    if 'PRESTIGE-GEN-XII' in list_response_model_name:
        return 'PRESTIGE-GEN-XII'
    elif 'PRESTIGE-GENXI' in list_response_model_name:
        return 'PRESTIGE-GENXI'
    else:
        return 'Gen10'

### Constants

In [4]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: 06_get_most_recent_app


### Make output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Import data

In [6]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/05_create_df/{str_filename}'
df = pd.read_parquet(str_uri)
# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


CPU times: user 4min 8s, sys: 7min 20s, total: 11min 29s
Wall time: 3min 38s


,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
0,588695.0,5712692,2021-07-26 16:15:47.424782700,1.0,5712692__7174945__20210719,5712692.0,7174945.0,1.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
1,588695.0,5712692,2021-07-26 16:15:47.424782700,0.0,5712692__7174946__20210719,5712692.0,7174946.0,0.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
2,588736.0,5702434,2021-07-26 16:29:29.390368600,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
3,588740.0,5713355,2021-07-26 16:31:38.506776200,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
4,588743.0,5713355,2021-07-26 16:32:04.321315800,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927681,NaN,8385049,2024-11-12 23:10:01+00:00,0.0,0__0__20241109,8385049.0,10354345.0,0.0,POST FALLS,Idaho,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,17294.0,Class 2
2927682,NaN,8387703,2024-11-12 23:45:43+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927683,NaN,8387703,2024-11-13 02:50:14+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927684,NaN,8388035,2024-11-13 04:01:11+00:00,1.0,8388035103580741,8388035.0,10358074.0,1.0,nan,Maryland,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,24495.0,Class 2


### Replace nan with Gen 10

In [7]:
df['RESPONSE_MODEL_NAME'] = df['RESPONSE_MODEL_NAME'].replace('nan', 'Gen10')

### Remove blackbook columns

In [8]:
list_cols = [col for col in df.columns if '__bb' in col.lower()]
df.drop(list_cols, axis=1, inplace=True)
# show
df

,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
0,588695.0,5712692,2021-07-26 16:15:47.424782700,1.0,5712692__7174945__20210719,5712692.0,7174945.0,1.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
1,588695.0,5712692,2021-07-26 16:15:47.424782700,0.0,5712692__7174946__20210719,5712692.0,7174946.0,0.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
2,588736.0,5702434,2021-07-26 16:29:29.390368600,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
3,588740.0,5713355,2021-07-26 16:31:38.506776200,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
4,588743.0,5713355,2021-07-26 16:32:04.321315800,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927681,NaN,8385049,2024-11-12 23:10:01+00:00,0.0,0__0__20241109,8385049.0,10354345.0,0.0,POST FALLS,Idaho,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,17294.0,Class 2
2927682,NaN,8387703,2024-11-12 23:45:43+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927683,NaN,8387703,2024-11-13 02:50:14+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927684,NaN,8388035,2024-11-13 04:01:11+00:00,1.0,8388035103580741,8388035.0,10358074.0,1.0,nan,Maryland,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,24495.0,Class 2


### Remove any accounts that applied to DLV1

In [9]:
%%time

df_tmp = df.groupby(by='ACCOUNTID', as_index=False).agg({
    'RESPONSE_MODEL_NAME': lambda x: list(x),
})
# tag
df_tmp['direct'] = df_tmp['RESPONSE_MODEL_NAME'].apply(
    lambda x: 1 if 'PRESTIGE-DLV1' in x else 0,
)
# get list of good accounts
list_accountid = list(df_tmp[df_tmp['direct'] == 0]['ACCOUNTID'])
# subset
df = df[df['ACCOUNTID'].isin(list_accountid)].copy()
# show
df

CPU times: user 58.7 s, sys: 58.6 s, total: 1min 57s
Wall time: 1min 57s


,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
0,588695.0,5712692,2021-07-26 16:15:47.424782700,1.0,5712692__7174945__20210719,5712692.0,7174945.0,1.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
1,588695.0,5712692,2021-07-26 16:15:47.424782700,0.0,5712692__7174946__20210719,5712692.0,7174946.0,0.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
2,588736.0,5702434,2021-07-26 16:29:29.390368600,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
3,588740.0,5713355,2021-07-26 16:31:38.506776200,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
4,588743.0,5713355,2021-07-26 16:32:04.321315800,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927681,NaN,8385049,2024-11-12 23:10:01+00:00,0.0,0__0__20241109,8385049.0,10354345.0,0.0,POST FALLS,Idaho,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,17294.0,Class 2
2927682,NaN,8387703,2024-11-12 23:45:43+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927683,NaN,8387703,2024-11-13 02:50:14+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927684,NaN,8388035,2024-11-13 04:01:11+00:00,1.0,8388035103580741,8388035.0,10358074.0,1.0,nan,Maryland,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,24495.0,Class 2


### Remove DLV1 apps

In [10]:
%%time

list_str_models = [
    'Gen10',
    'PRESTIGE-GENXI',
    'PRESTIGE-GEN-XII',
]
df = df[df['RESPONSE_MODEL_NAME'].isin(list_str_models)].copy()
# show
df

CPU times: user 57.5 s, sys: 51 s, total: 1min 48s
Wall time: 1min 48s


,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
0,588695.0,5712692,2021-07-26 16:15:47.424782700,1.0,5712692__7174945__20210719,5712692.0,7174945.0,1.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
1,588695.0,5712692,2021-07-26 16:15:47.424782700,0.0,5712692__7174946__20210719,5712692.0,7174946.0,0.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
2,588736.0,5702434,2021-07-26 16:29:29.390368600,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
3,588740.0,5713355,2021-07-26 16:31:38.506776200,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
4,588743.0,5713355,2021-07-26 16:32:04.321315800,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927681,NaN,8385049,2024-11-12 23:10:01+00:00,0.0,0__0__20241109,8385049.0,10354345.0,0.0,POST FALLS,Idaho,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,17294.0,Class 2
2927682,NaN,8387703,2024-11-12 23:45:43+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927683,NaN,8387703,2024-11-13 02:50:14+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927684,NaN,8388035,2024-11-13 04:01:11+00:00,1.0,8388035103580741,8388035.0,10358074.0,1.0,nan,Maryland,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,24495.0,Class 2


### Convert request time to datetime (mountain time)

In [11]:
%%time

# make dtm - utc
df['REQUEST_DATETIME'] = pd.to_datetime(
    df['REQUEST_DATETIME'],
    utc=True,
    errors='coerce',
)
# Specify Mountain Time timezone
tz = pytz.timezone('America/Denver')
# convert to mountain
df['REQUEST_DATETIME'] = df['REQUEST_DATETIME'].dt.tz_convert(tz)
# show
df

CPU times: user 6.86 s, sys: 3.26 s, total: 10.1 s
Wall time: 10.3 s


,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
0,588695.0,5712692,2021-07-26 10:15:47.424782700-06:00,1.0,5712692__7174945__20210719,5712692.0,7174945.0,1.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
1,588695.0,5712692,2021-07-26 10:15:47.424782700-06:00,0.0,5712692__7174946__20210719,5712692.0,7174946.0,0.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
2,588736.0,5702434,2021-07-26 10:29:29.390368600-06:00,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
3,588740.0,5713355,2021-07-26 10:31:38.506776200-06:00,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
4,588743.0,5713355,2021-07-26 10:32:04.321315800-06:00,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927681,NaN,8385049,2024-11-12 16:10:01-07:00,0.0,0__0__20241109,8385049.0,10354345.0,0.0,POST FALLS,Idaho,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,17294.0,Class 2
2927682,NaN,8387703,2024-11-12 16:45:43-07:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927683,NaN,8387703,2024-11-12 19:50:14-07:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927684,NaN,8388035,2024-11-12 21:01:11-07:00,1.0,8388035103580741,8388035.0,10358074.0,1.0,nan,Maryland,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,24495.0,Class 2


### Get Gen 12 payload, then Gen 11, then Gen 10

In [12]:
df_tmp = df.groupby(by='ACCOUNTID', as_index=False).agg({
    'RESPONSE_MODEL_NAME': lambda x: list(dict.fromkeys(list(x))),
})
df_tmp['model'] = df_tmp['RESPONSE_MODEL_NAME'].apply(yield_to_model)
df_tmp['account_model'] = df_tmp['ACCOUNTID'].astype(str) + ' - ' + df_tmp['model']
# make list so we can use isin
list_account_model = list(df_tmp['account_model'])
# save memory
del df_tmp

# create account_model column
df['account_model'] = df['ACCOUNTID'].astype(str) + ' - ' + df['RESPONSE_MODEL_NAME']
# subset
df = df[df['account_model'].isin(list_account_model)].copy()
# show
df

,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app,account_model
0,588695.0,5712692,2021-07-26 10:15:47.424782700-06:00,1.0,5712692__7174945__20210719,5712692.0,7174945.0,1.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5712692 - Gen10
1,588695.0,5712692,2021-07-26 10:15:47.424782700-06:00,0.0,5712692__7174946__20210719,5712692.0,7174946.0,0.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5712692 - Gen10
2,588736.0,5702434,2021-07-26 10:29:29.390368600-06:00,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5702434 - Gen10
3,588740.0,5713355,2021-07-26 10:31:38.506776200-06:00,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5713355 - Gen10
4,588743.0,5713355,2021-07-26 10:32:04.321315800-06:00,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5713355 - Gen10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927673,NaN,8381854,2024-11-12 18:12:21-07:00,1.0,8381854103504821,8381854.0,10350482.0,1.0,nan,Pennsylvania,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,16349.0,Class 1,8381854 - PRESTIGE-GEN-XII
2927675,NaN,8382944,2024-11-12 15:51:29-07:00,1.0,8382944103517881,8382944.0,10351788.0,1.0,nan,Alabama,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,15000.0,Class 3,8382944 - PRESTIGE-GEN-XII
2927677,NaN,8383985,2024-11-12 16:08:19-07:00,1.0,8383985103530551,8383985.0,10353055.0,1.0,nan,New Jersey,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,17988.0,Class 3,8383985 - PRESTIGE-GEN-XII
2927684,NaN,8388035,2024-11-12 21:01:11-07:00,1.0,8388035103580741,8388035.0,10358074.0,1.0,nan,Maryland,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,24495.0,Class 2,8388035 - PRESTIGE-GEN-XII


### Get most recent request by account

In [13]:
# get max request date by account
df_tmp = df.groupby(by='ACCOUNTID', as_index=False).agg({
    'REQUEST_DATETIME': 'max',
})
df_tmp['account_date'] = df_tmp['ACCOUNTID'].astype(str) + ' - ' + df_tmp['REQUEST_DATETIME'].astype(str)
# make list so we can use isin
list_account_date = list(df_tmp['account_date'])
# save memory
del df_tmp

# create account_model column
df['account_date'] = df['ACCOUNTID'].astype(str) + ' - ' + df['REQUEST_DATETIME'].astype(str)
# subset
df = df[df['account_date'].isin(list_account_date)].copy()
# show
df

,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app,account_model,account_date
2,588736.0,5702434,2021-07-26 10:29:29.390368600-06:00,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5702434 - Gen10,5702434 - 2021-07-26 10:29:29.390368600-06:00
9,588761.0,5714239,2021-07-26 10:39:34.112102500-06:00,1.0,5714239__7176826__20210720,5714239.0,7176826.0,1.0,W VALLEY CITY,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5714239 - Gen10,5714239 - 2021-07-26 10:39:34.112102500-06:00
10,588776.0,5713063,2021-07-26 10:48:39.321110400-06:00,1.0,5713063__7175396__20210719,5713063.0,7175396.0,1.0,CHICAGO,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5713063 - Gen10,5713063 - 2021-07-26 10:48:39.321110400-06:00
34,589470.0,5709088,2021-07-27 06:24:35.219815100-06:00,1.0,5709088__7170547__20210715,5709088.0,7170547.0,1.0,SAINT LOUIS,Missouri,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5709088 - Gen10,5709088 - 2021-07-27 06:24:35.219815100-06:00
45,589491.0,5702106,2021-07-27 06:40:50.011549800-06:00,1.0,5702106__7162085__20210707,5702106.0,7162085.0,1.0,SACRAMENTO,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,5702106 - Gen10,5702106 - 2021-07-27 06:40:50.011549800-06:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927641,NaN,8388060,2024-11-12 18:03:51-07:00,1.0,8388060103581051,8388060.0,10358105.0,1.0,nan,Texas,...,-1.0,-1.0,-3.0,-3.0,-3.0,-5.0,23400.00,Class 1,8388060 - PRESTIGE-GEN-XII,8388060 - 2024-11-12 18:03:51-07:00
2927645,NaN,8317817,2024-11-12 19:11:15-07:00,1.0,8317817102737051,8317817.0,10273705.0,1.0,nan,North Carolina,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,27967.00,Class 1,8317817 - PRESTIGE-GEN-XII,8317817 - 2024-11-12 19:11:15-07:00
2927646,NaN,8337035,2024-11-12 20:01:18-07:00,1.0,8337035102967661,8337035.0,10296766.0,1.0,nan,Virginia,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,13597.31,Class 3,8337035 - PRESTIGE-GEN-XII,8337035 - 2024-11-12 20:01:18-07:00
2927659,NaN,8370375,2024-11-12 20:24:26-07:00,1.0,8370375103369011,8370375.0,10336901.0,1.0,nan,Tennessee,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,31201.09,Class 1,8370375 - PRESTIGE-GEN-XII,8370375 - 2024-11-12 20:24:26-07:00


### Drop columns

In [14]:
list_cols = [
    'account_model',
    'account_date',
]
df.drop(list_cols, axis=1, inplace=True)
# show
df

,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
2,588736.0,5702434,2021-07-26 10:29:29.390368600-06:00,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
9,588761.0,5714239,2021-07-26 10:39:34.112102500-06:00,1.0,5714239__7176826__20210720,5714239.0,7176826.0,1.0,W VALLEY CITY,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
10,588776.0,5713063,2021-07-26 10:48:39.321110400-06:00,1.0,5713063__7175396__20210719,5713063.0,7175396.0,1.0,CHICAGO,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
34,589470.0,5709088,2021-07-27 06:24:35.219815100-06:00,1.0,5709088__7170547__20210715,5709088.0,7170547.0,1.0,SAINT LOUIS,Missouri,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
45,589491.0,5702106,2021-07-27 06:40:50.011549800-06:00,1.0,5702106__7162085__20210707,5702106.0,7162085.0,1.0,SACRAMENTO,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927641,NaN,8388060,2024-11-12 18:03:51-07:00,1.0,8388060103581051,8388060.0,10358105.0,1.0,nan,Texas,...,-1.0,-1.0,-1.0,-1.0,-3.0,-3.0,-3.0,-5.0,23400.00,Class 1
2927645,NaN,8317817,2024-11-12 19:11:15-07:00,1.0,8317817102737051,8317817.0,10273705.0,1.0,nan,North Carolina,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,27967.00,Class 1
2927646,NaN,8337035,2024-11-12 20:01:18-07:00,1.0,8337035102967661,8337035.0,10296766.0,1.0,nan,Virginia,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,13597.31,Class 3
2927659,NaN,8370375,2024-11-12 20:24:26-07:00,1.0,8370375103369011,8370375.0,10336901.0,1.0,nan,Tennessee,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,31201.09,Class 1


### Fill NaN

In [15]:
# find duplicate column names
list_cols = [col.lower() for col in df.columns]
ser_val_counts = pd.Series(list_cols).value_counts()
list_cols_dups = list(ser_val_counts[ser_val_counts==2].index)
# get the lower and upper
list_cols_dups = [col for col in df.columns if col.lower() in list_cols_dups]
# get lower
list_cols_lower = [col for col in list_cols_dups if col.islower()]
# get non lower
list_cols_not_lower = [col for col in list_cols_dups if not col.islower()]
# make dict
dict_fillna = dict(zip(list_cols_lower, list_cols_not_lower))
# iterate and fill
for key, val in tqdm(dict_fillna.items()):
    # get dtype
    str_type = df[key].dtype
    # logic
    if str_type not in ['int64','float64']:
        # replace
        df[key] = df[key].replace(['NaN', 'nan', 'None'], np.nan)
    else:
        pass
    df[key] = df[key].fillna(df[val])
# drop
df.drop(list_cols_not_lower, axis=1, inplace=True)
# show
df

100%|██████████| 83/83 [00:58<00:00,  1.41it/s]


,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
2,588736.0,5702434,2021-07-26 10:29:29.390368600-06:00,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
9,588761.0,5714239,2021-07-26 10:39:34.112102500-06:00,1.0,5714239__7176826__20210720,5714239.0,7176826.0,1.0,W VALLEY CITY,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
10,588776.0,5713063,2021-07-26 10:48:39.321110400-06:00,1.0,5713063__7175396__20210719,5713063.0,7175396.0,1.0,CHICAGO,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
34,589470.0,5709088,2021-07-27 06:24:35.219815100-06:00,1.0,5709088__7170547__20210715,5709088.0,7170547.0,1.0,SAINT LOUIS,Missouri,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
45,589491.0,5702106,2021-07-27 06:40:50.011549800-06:00,1.0,5702106__7162085__20210707,5702106.0,7162085.0,1.0,SACRAMENTO,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927641,NaN,8388060,2024-11-12 18:03:51-07:00,1.0,8388060103581051,8388060.0,10358105.0,1.0,NaN,Texas,...,-1.0,-1.0,-1.0,-1.0,-3.0,-3.0,-3.0,-5.0,23400.00,Class 1
2927645,NaN,8317817,2024-11-12 19:11:15-07:00,1.0,8317817102737051,8317817.0,10273705.0,1.0,NaN,North Carolina,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,27967.00,Class 1
2927646,NaN,8337035,2024-11-12 20:01:18-07:00,1.0,8337035102967661,8337035.0,10296766.0,1.0,NaN,Virginia,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,13597.31,Class 3
2927659,NaN,8370375,2024-11-12 20:24:26-07:00,1.0,8370375103369011,8370375.0,10336901.0,1.0,NaN,Tennessee,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,31201.09,Class 1


### Lower columns

In [16]:
list_cols = [col.lower() for col in df.columns]
df.columns = list_cols
# show
df

,bigdoveid,accountid,request_datetime,bitdebtor,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
2,588736.0,5702434,2021-07-26 10:29:29.390368600-06:00,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
9,588761.0,5714239,2021-07-26 10:39:34.112102500-06:00,1.0,5714239__7176826__20210720,5714239.0,7176826.0,1.0,W VALLEY CITY,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
10,588776.0,5713063,2021-07-26 10:48:39.321110400-06:00,1.0,5713063__7175396__20210719,5713063.0,7175396.0,1.0,CHICAGO,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
34,589470.0,5709088,2021-07-27 06:24:35.219815100-06:00,1.0,5709088__7170547__20210715,5709088.0,7170547.0,1.0,SAINT LOUIS,Missouri,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
45,589491.0,5702106,2021-07-27 06:40:50.011549800-06:00,1.0,5702106__7162085__20210707,5702106.0,7162085.0,1.0,SACRAMENTO,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927641,NaN,8388060,2024-11-12 18:03:51-07:00,1.0,8388060103581051,8388060.0,10358105.0,1.0,NaN,Texas,...,-1.0,-1.0,-1.0,-1.0,-3.0,-3.0,-3.0,-5.0,23400.00,Class 1
2927645,NaN,8317817,2024-11-12 19:11:15-07:00,1.0,8317817102737051,8317817.0,10273705.0,1.0,NaN,North Carolina,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,27967.00,Class 1
2927646,NaN,8337035,2024-11-12 20:01:18-07:00,1.0,8337035102967661,8337035.0,10296766.0,1.0,NaN,Virginia,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,13597.31,Class 3
2927659,NaN,8370375,2024-11-12 20:24:26-07:00,1.0,8370375103369011,8370375.0,10336901.0,1.0,NaN,Tennessee,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,31201.09,Class 1


### Sort

In [17]:
list_cols = [
    'request_datetime',
    'accountid',
    'bitdebtor',
]
list_bool = [
    True,
    True,
    False,
]
df.sort_values(by=list_cols, ascending=list_bool, inplace=True)
# show
df

,bigdoveid,accountid,request_datetime,bitdebtor,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
2,588736.0,5702434,2021-07-26 10:29:29.390368600-06:00,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
9,588761.0,5714239,2021-07-26 10:39:34.112102500-06:00,1.0,5714239__7176826__20210720,5714239.0,7176826.0,1.0,W VALLEY CITY,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
10,588776.0,5713063,2021-07-26 10:48:39.321110400-06:00,1.0,5713063__7175396__20210719,5713063.0,7175396.0,1.0,CHICAGO,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
89,589084.0,5713732,2021-07-27 03:02:35.330097400-06:00,1.0,5713732__7176216__20210720,5713732.0,7176216.0,1.0,OAK PARK,Michigan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
96,589108.0,5715634,2021-07-27 03:18:12.219009700-06:00,1.0,5715634__7178525__20210722,5715634.0,7178525.0,1.0,Goodyear,Arizona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927104,NaN,8373613,2024-11-12 22:56:11-07:00,1.0,8373613103407211,8373613.0,10340721.0,1.0,NaN,California,...,-1.0,-1.0,-1.0,-1.0,0.0,0.0,0.0,-5.0,29009.0,Class 3
2927422,NaN,8365714,2024-11-12 22:57:22-07:00,1.0,8365714103312141,8365714.0,10331214.0,1.0,NaN,Michigan,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,30999.0,Class 3
2927423,NaN,8365714,2024-11-12 22:57:22-07:00,0.0,8365714103312150,8365714.0,10331215.0,0.0,NaN,Michigan,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,30999.0,Class 3
2927374,NaN,8372495,2024-11-12 22:57:37-07:00,1.0,8372495103393811,8372495.0,10339381.0,1.0,NaN,Utah,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,20743.0,Class 3


### Write to s3

In [18]:
# convert dtypes
for col in tqdm(df.columns):
    # get dtype
    str_dtype = df[col].dtype
    # logic
    if str_dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)
    else:
        pass

100%|██████████| 3281/3281 [00:39<00:00, 82.33it/s]  


In [19]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 42.3 s, sys: 247 ms, total: 42.5 s
Wall time: 47.1 s
